# TonyPi 基础功能测试

本 Notebook 根据同目录的 `demo.py` 编写，供初学者逐项测试机器人：站立、拍照、转头和行走。请按顺序运行；每个功能都在独立单元中，确认机器人周围安全后再执行。

## 1. 初始化

先运行此单元，加载动作组、相机和舵机控制所需的库。若这里报错，请确认机器人本体已启动，并在 TonyPi 环境中打开本 Notebook。

In [ ]:
import time
import subprocess

import hiwonder.ActionGroupControl as AGC
import hiwonder.ros_robot_controller_sdk as rrc
from hiwonder.Controller import Controller

board = rrc.Board()
ctl = Controller(board)
print('初始化完成')

## 2. 定义拍照与转头函数

运行本单元后，后面的测试单元才可以调用这些函数。拍摄的照片会保存到机器人 `/home/pi/Pictures/` 目录。头部水平舵机的脉宽 `1500` 为正前方；数值变大向左、变小向右。

In [ ]:
def capture_image():
    """拍照并返回保存路径。"""
    timestamp = int(time.time())
    filename = f'/home/pi/Pictures/photo_{timestamp}.jpg'
    cmd = f'fswebcam -r 2592x1944 --no-banner -S 3 {filename}'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    if result.returncode != 0:
        print(f'拍照失败: {result.stderr}')
        return None

    print(f'照片已保存: {filename}')
    return filename


def turn_right_30():
    """头部向右转约 30 度。"""
    ctl.set_pwm_servo_pulse(2, 1200, 1000)
    time.sleep(2)


def turn_left_30():
    """头部向左转约 30 度。"""
    ctl.set_pwm_servo_pulse(2, 1800, 1000)
    time.sleep(2)


def turn_ahead():
    """头部回到正前方。"""
    ctl.set_pwm_servo_pulse(2, 1500, 1000)
    time.sleep(2)

## 3. 测试站立

所有动作测试前建议先站立。请将机器人放在平整地面，双脚周围不要有电线或障碍物。

In [ ]:
AGC.runActionGroup('stand')
print('已执行站立动作')

## 4. 测试拍照

运行后等待数秒，输出中会显示照片保存位置。

In [ ]:
filename = capture_image()

## 5. 测试向左转头

运行后，机器人头部向左转约 30 度。

In [ ]:
turn_left_30()

## 6. 测试向右转头

运行后，机器人头部向右转约 30 度。

In [ ]:
turn_right_30()

## 7. 头部回正

完成转头测试后运行，让机器人面向正前方。

In [ ]:
turn_ahead()

## 8. 测试原地转向

请确保机器人四周至少留出 50 厘米空间。`times=1` 表示只执行一次小步右转。

In [ ]:
AGC.runActionGroup('turn_right', times=1)
AGC.runActionGroup('stand')

## 9. 测试向前行走

请确保机器人前方至少有 1 米平整空间。第一次建议保持 `times=1`；确认稳定后再逐步增加次数。运行后会自动站立收尾。

In [ ]:
AGC.runActionGroup('go_forward_fast', times=1, with_stand=True)
print('行走测试完成')

## 10. 连续演示（可选）

确认前面每项都正常后，再运行这个完整演示：站立、拍照、左转头拍照、右转头拍照、回正、原地转向、向前走。

In [ ]:
AGC.runActionGroup('stand')
capture_image()
turn_left_30()
capture_image()
turn_right_30()
capture_image()
turn_ahead()
AGC.runActionGroup('turn_right', times=1)
AGC.runActionGroup('go_forward_fast', times=1, with_stand=True)
print('完整演示结束')